
# Hands-On – Kernels and Optimization
*(20 min, demo-only)*


---

## Part 1  What Is a Kernel?

A **kernel** is a GPU function (marked `__global__`) launched by the CPU but executed in parallel by many threads.

- Each thread executes the same code on different data.
- Threads are grouped into **blocks** (work-groups).
- Each GPU Compute Unit (CU) runs multiple **wavefronts** (64 threads on AMD).

### Execution Hierarchy

```
Host → launch kernel → GPU executes:
   Grid
   ├── Block 0 (e.g., 256 threads)
   │     ├── Wavefront 0 (64 threads)
   │     ├── Wavefront 1 (64 threads)
   └── Block 1 (e.g., 256 threads)
```

**Launch Example:**

```cpp
myKernel<<<gridDim, blockDim>>>(args...);
```


Now that we know what a kernel is and how to launch one, let’s look at how memory access patterns alone can make or break performance.


---

## Part 2  How to Write a HIP Kernel (Step-by-Step)

This part walks through building and running a minimal HIP kernel. The audience does not need to type; the presenter runs the cells.

#### Learning Objectives
By the end of this part, you will be able to:
- Understand the structure of a HIP kernel and launch configuration
- Implement and validate a simple elementwise kernel (vector add)
- Prepare for more advanced optimizations like coalescing/unrolling in Part 3



### Step 1 — Define the Kernel

```cpp
// File: vector_add.cpp
#include <hip/hip_runtime.h>
#include <iostream>
#include <cmath>

__global__ void vectorAdd(const float* A, const float* B, float* C, int N) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    if (gid < N)
        C[gid] = A[gid] + B[gid];
}
```



### Step 2 — Allocate and Copy Memory

```cpp
#define HIP_CHECK(cmd) do { hipError_t e = cmd; if(e != hipSuccess){     std::cerr << "HIP error: " << hipGetErrorString(e) << std::endl; exit(EXIT_FAILURE);} } while(0)

int main() {
    const int N = 1 << 20;            // 1 M elements
    const size_t bytes = N * sizeof(float);

    // Allocate host memory
    float *hA = new float[N], *hB = new float[N], *hC = new float[N];
    for (int i = 0; i < N; ++i) { hA[i] = i * 0.001f; hB[i] = i * 0.002f; }

    // Allocate device memory
    float *dA, *dB, *dC;
    HIP_CHECK(hipMalloc(&dA, bytes));
    HIP_CHECK(hipMalloc(&dB, bytes));
    HIP_CHECK(hipMalloc(&dC, bytes));

    // Copy inputs to device
    HIP_CHECK(hipMemcpy(dA, hA, bytes, hipMemcpyHostToDevice));
    HIP_CHECK(hipMemcpy(dB, hB, bytes, hipMemcpyHostToDevice));
```



### Step 3 — Launch the Kernel

```cpp
    int blockSize = 256;
    int gridSize  = (N + blockSize - 1) / blockSize;
    hipLaunchKernelGGL(vectorAdd, dim3(gridSize), dim3(blockSize), 0, 0, dA, dB, dC, N);
    HIP_CHECK(hipDeviceSynchronize());
```



### Step 4 — Copy Results and Verify

```cpp
    HIP_CHECK(hipMemcpy(hC, dC, bytes, hipMemcpyDeviceToHost));

    bool correct = true;
    for (int i = 0; i < N; ++i) {
        float expected = hA[i] + hB[i];
        if (fabs(hC[i] - expected) > 1e-5) { correct = false; break; }
    }

    std::cout << (correct ? "Results verified - CORRECT!\n"
                          : "Verification FAILED!\n");

    delete[] hA; delete[] hB; delete[] hC;
    HIP_CHECK(hipFree(dA)); HIP_CHECK(hipFree(dB)); HIP_CHECK(hipFree(dC));
    return 0;
```



### Full Example Source (for quick run)

The following cell writes a complete example to `vector_add.cpp` so you can compile and execute it immediately.


In [1]:
%%writefile vector_add.cpp

#include <hip/hip_runtime.h>
#include <iostream>
#include <cmath>

#define HIP_CHECK(cmd) do { hipError_t e = cmd; if(e != hipSuccess){ \
    std::cerr << "HIP error: " << hipGetErrorString(e) << std::endl; exit(EXIT_FAILURE);} } while(0)

__global__ void vectorAdd(const float* A, const float* B, float* C, int N) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    if (gid < N)
        C[gid] = A[gid] + B[gid];
}

int main() {
    const int N = 1 << 20; // 1M elements
    const size_t bytes = N * sizeof(float);

    float *hA = new float[N], *hB = new float[N], *hC = new float[N];
    for (int i = 0; i < N; ++i) { hA[i] = i * 0.001f; hB[i] = i * 0.002f; }

    float *dA, *dB, *dC;
    HIP_CHECK(hipMalloc(&dA, bytes));
    HIP_CHECK(hipMalloc(&dB, bytes));
    HIP_CHECK(hipMalloc(&dC, bytes));

    HIP_CHECK(hipMemcpy(dA, hA, bytes, hipMemcpyHostToDevice));
    HIP_CHECK(hipMemcpy(dB, hB, bytes, hipMemcpyHostToDevice));

    int blockSize = 256;
    int gridSize = (N + blockSize - 1) / blockSize;
    hipLaunchKernelGGL(vectorAdd, dim3(gridSize), dim3(blockSize), 0, 0, dA, dB, dC, N);
    HIP_CHECK(hipDeviceSynchronize());

    HIP_CHECK(hipMemcpy(hC, dC, bytes, hipMemcpyDeviceToHost));

    bool correct = true;
    for (int i = 0; i < N; ++i) {
        float expected = hA[i] + hB[i];
        if (fabs(hC[i] - expected) > 1e-5) { correct = false; break; }
    }
    std::cout << (correct ? "Results verified - CORRECT!\n"
                          : "Verification FAILED!\n");

    delete[] hA; delete[] hB; delete[] hC;
    HIP_CHECK(hipFree(dA)); HIP_CHECK(hipFree(dB)); HIP_CHECK(hipFree(dC));
    return 0;
}

Overwriting vector_add.cpp


### Compile

In [2]:
!hipcc -O3 vector_add.cpp -o vector_add
!echo "Built: ./vector_add"
!ls -lh vector_add* || true

Built: ./vector_add
-rwxr-xr-x 1 root root  22K Nov 25 01:53 vector_add
-rw-r--r-- 1 root root 1.7K Nov 25 01:53 vector_add.cpp


### Run

In [3]:
!./vector_add || echo "[WARN] Failed to run ./vector_add"

Results verified - CORRECT!



**Concept Recap**

- Each thread processes one element.
- Use `blockIdx`, `threadIdx`, and `blockDim` to compute a unique global index.
- Always check boundaries (`gid < N`) and verify results.



---

## Part 3  How to Optimize a Kernel

We will structure optimization into three steps:
1. **Step 1 – How to Profile Performance (NVIDIA vs AMD; we use rocprof-compute here)**
2. **Step 2 – Common Optimization Techniques**
3. **Step 3 – Optimization Case Studies (two real examples)**



### Step 1 — How to Profile Performance (NVIDIA vs AMD)

**AMD (ROCm / HIP):**
- [**rocprof-compute**](https://rocm.docs.amd.com/projects/rocprofiler-compute/en/latest/) — kernel-level metrics & summaries (recommended for HIP compute analysis).
- [**rocprofv3**](https://rocm.docs.amd.com/projects/rocprofiler-sdk/en/latest/how-to/using-rocprofv3.html) — tracing & stats; suitable for quick stats or timeline capture.
- **rocminfo**, **rocm-smi** — device information and runtime status.
- Typical usage:
  - `rocprof-compute --kernels --summary ./your_binary`
  - `rocprofv3 --stats ./your_binary`

**NVIDIA (CUDA):**
- **Nsight Compute (`ncu`)** — per-kernel metrics (occupancy, memory transactions, warp efficiency).
- **Nsight Systems (`nsys`)** — timeline tracing, CPU–GPU overlap, system-wide view.
- Typical usage:
  - `ncu --set full --target-processes all ./your_binary`
  - `nsys profile -t cuda,osrt -o trace ./your_binary`


### Step 2 — Common Optimization Techniques

- **Memory Access**
  - Prefer sequential, coalesced access; avoid large strides/scatter.
  - Exploit locality; use shared memory/tiling where beneficial.
- **Control Flow**
  - Minimize warp/wavefront divergence; align branches to 64-thread wavefronts.
  - Use predication for short alternative paths; branch on uniform conditions.
- **Occupancy & Resources**
  - Balance registers and shared memory to allow more concurrent wavefronts.
  - Choose sensible block sizes (multiples of 64 on AMD).
- **Bank Conflicts**
  - Lay out shared memory to avoid bank conflicts; add padding if needed.
- **Algorithmic Structure**
  - Partition data or launch separate kernels for heterogeneous work.
  - Reduce synchronization and atomics where possible.
- **Measure, then iterate**
  - Always validate speedups with profiling; avoid over-optimizing cold paths.



### Step 3 — Optimization Case Studies



### Case A: Memory Coalescing

#### 1. Experiment Description
We compare two HIP kernels that both perform the same amount of work:

- `uncoalesced_kernel`: each thread reads `ELEMENTS_PER_THREAD` elements
  with a **strided / non-coalesced** access pattern.
- `coalesced_kernel`: each thread reads the same number of elements but
  with a **coalesced** access pattern, so threads in the same wavefront
  access contiguous addresses at each iteration.

Both kernels produce the same numerical result (each output is `256`), so
the difference is purely due to the memory access pattern.

#### 2. Data volume

- `NUM_ELEMENTS = 33,554,432` float inputs
- Input bytes: `33,554,432 × 4 = 134,217,728` bytes
- Output bytes: `131,072 × 4 = 524,288` bytes
- Total bytes (approx):  
  `134,742,016 ≈ 0.1347 GB`

#### 3. Run demo

In [4]:
!hipcc -O3 -g -o ./01-memory-coalescing/coalescing_naive ./01-memory-coalescing/coalescing_naive.cpp
!./01-memory-coalescing/coalescing_naive

Running uncoalesced_kernel
  NUM_THREADS_TOTAL   = 131072
  ELEMENTS_PER_THREAD = 256
  NUM_ELEMENTS        = 33554432
Average time over 50 runs: 0.489687 ms
Sample output[0] = 256 (expected 256)


In [5]:
!hipcc -O3 -g -o ./01-memory-coalescing/coalescing_optimized ./01-memory-coalescing/coalescing_optimized.cpp
!./01-memory-coalescing/coalescing_optimized

Running coalesced_kernel
  NUM_THREADS_TOTAL   = 131072
  ELEMENTS_PER_THREAD = 256
  NUM_ELEMENTS        = 33554432
Average time over 50 runs: 0.0675296 ms
Sample output[0] = 256 (expected 256)


#### 4. Timing & effective bandwidth

Measured on MI300X using `hipEventRecord` timing (average over 50 runs):

| Kernel Variant          | Time (ms) | Speedup vs Naive | Effective Bandwidth (GB/s) |
|-------------------------|----------:|-----------------:|---------------------------:|
| Uncoalesced (naive)     | 0.487586  | 1.0×             | ~276 GB/s                  |
| Coalesced (optimized)   | 0.0675114 | **7.22×**        | ~1,996 GB/s                |


> The two kernels are functionally identical (both produce `256` for
> `output[0]`), but the coalesced version achieves about **7.2× speedup**
> and pushes the effective memory bandwidth from ~276 GB/s to almost
> 2 TB/s. This clearly illustrates how a purely structural change in the
> access pattern (coalesced vs strided) can dominate performance on a
> modern GPU.

![output.png](./01-memory-coalescing/output.png)

We’ve seen how where you read from memory matters. Next, we’ll see how how often you branch/loop can matter just as much – with a dramatic loop-unrolling example.


### Case B: Loop Unrolling
#### 1. Experiment Description

We compare two HIP kernels that compute a 3×3 convolution on a **4096×4096** image:

1. **Naive version**  
   - Uses two nested loops (`for ky` and `for kx`)  
   - Uses `#pragma unroll 1` to **force** the compiler *not* to unroll  
   - Repeats the convolution **INNER_ITERS = 64** times per pixel to make loop overhead visible

2. **Unrolled version**  
   - Fully manually unrolls the 3×3 convolution (9 FMA operations)  
   - Also repeats it same INNER_ITERS times  
   - Reduces loop overhead and improves ILP (Instruction-Level Parallelism)

Both kernels compute identical numerical output (`16` for the Gaussian kernel on a constant image), ensuring functional equivalence.


#### 2. FLOPs Calculation

For a 3×3 convolution:

- Each convolution = **9 FMAs = 18 FLOPs**
- Number of valid pixels (no padding):  
  \[
  (4096 - 2)^2 = 4094^2 = 16,760,836
  \]
- Repeated `INNER_ITERS = 64` times per pixel  
- Total FLOPs:
  \[
  \text{FLOPs} = 16,760,836 \times 18 \times 64 \approx 1.93 \times 10^{10}
  \]


#### 3. Run demo

In [6]:
!hipcc -O3 -g -o ./02-loop-unrolling/conv3x3_naive ./02-loop-unrolling/conv3x3_naive.cpp
!./02-loop-unrolling/conv3x3_naive

Running conv3x3_naive_no_unroll on 4096x4096 image, INNER_ITERS=64
Average time: 3.62858 ms
Sample output = 16


In [7]:
!hipcc -O3 -g -o ./02-loop-unrolling/conv3x3_unrolled ./02-loop-unrolling/conv3x3_unrolled.cpp
!./02-loop-unrolling/conv3x3_unrolled

Running conv3x3_unrolled on 4096x4096 image, INNER_ITERS=64
Average time: 0.0605785 ms
Sample output = 16



#### 4. Timing Results

Measured on an MI300X container environment using `hipEventRecord` timing.

| Kernel Variant                      | Time (ms) | Speedup vs Naive | Approx Throughput |
|------------------------------------|----------:|-----------------:|-------------------:|
| **Naive (looped, unroll disabled)** | 3.62491 ms | 1.0×             | ~5.3 TFLOP/s        |
| **Manually unrolled (no loops)**    | 0.06039 ms | **60.0×**        | ~320 TFLOP/s*       |

**\*** *Absolute TFLOP/s is inflated because the kernel is extremely short; event timing + launch overhead dominate.  
The relative speedup (≈60×) is the meaningful takeaway.*

![output.png](./02-loop-unrolling/output.png)

With both memory coalescing and loop unrolling in mind, you have two of the most impactful tools for GPU kernel optimization.

---

## Summary

This notebook provided a comprehensive walkthrough of **manual GPU kernel engineering** — from fundamentals to optimization and profiling.

1. **Execution Model**  
   Understand how grids, blocks, and threads map to GPU hardware, and how wavefronts (AMD) or warps (NVIDIA) execute in lockstep.


3. **Profiling and Iteration**  
   Use tools like `rocprofv3` or `rocprof-compute` to analyze bandwidth, occupancy, register pressure, and LDS usage.  
   Iterate based on data-driven insights.


2. **Memory Optimization**  
   - **Memory Coalescing**: Ensure adjacent threads access adjacent memory addresses.  
   - **Shared Memory / Tiling**: Cache data blocks locally to reduce global memory traffic.  


4. **Control Flow Optimization**  
    - **Loop Unrolling**: Remove loop-control overhead, reduce index recomputation, and expose longer straight-line instruction sequences.  


5. **Hands-on Results**  
   The Naive vs. Tiled Matrix Multiplication experiment demonstrated the impact of shared-memory reuse, often yielding **10–40× speedups**.

> **Key Takeaway:** GPU kernel optimization is a process of measurement and refinement — combining hardware insight, memory behavior, and profiling feedback to iteratively improve performance.

**Next Steps:**
- Use `rocprof-compute` for deeper instruction and roofline analysis.  
- Explore other optimization methods and examples.  
- Consider how low-level optimization principles can inform **automatic kernel generation** and **model-level optimization**.
